# 第 1 周 · 第 2 天作业 —— 本地 Ollama 由主题行生成邮件

## 练习目标（理念）

和第 1 天作业同一目标——根据 **邮件主题行** 生成格式化邮件——但后端换成**本地 Ollama**，不再调用云端 OpenAI。

你会练习到：

1. 自定义 **system prompt** 与 **user prompt 前缀**
2. 按 **OpenAI 兼容** 的 Ollama `/v1` 接口组装 `messages`
3. 用本地模型（如 `llama3.2`）根据 `email_subject` 生成邮件
4. 把返回的 Markdown 在笔记本里展示出来

## 和本课 Day 2 的关系

| 本课概念 | 本作业里你会看到 |
|----------|------------------|
| Ollama + OpenAI 兼容客户端 | `OpenAI(base_url=..., api_key='ollama')` |
| 本地模型名 | `model="llama3.2"`（需事先 `ollama pull`） |
| 同一套 `messages` 形状 | system / user 与云端调用写法一致 |
| 无需云端密钥 | 本格未使用 `OPENAI_API_KEY` / `load_dotenv` |

## 怎么跑

1. 本机启动 Ollama，并确保已拉取 `llama3.2`
2. 从上到下运行单元格；可改 `email_subject` 再跑，对比不同主题下的本地模型输出


In [ ]:
# ========== 第 1 步：导入库并写好提示词 ==========

# 导入标准库 os：本格虽未读密钥，保留原导入以便与 Day1 结构对照
import os
# 从 IPython.display 导入 Markdown / display：在笔记本里渲染模型输出
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：这里用来对接 Ollama 的 OpenAI 兼容接口
from openai import OpenAI

# system prompt：定模型角色——擅长根据主题行写邮件；保留英文以免改变行为
system_prompt = "You are an expert at generating email contents from subject lines"
# user prompt 前缀：说明期望输出格式；真正主题行会拼在后面
user_prompt_prefix = """
    Generate a formatted email message with bullet lists
    and other details given the subject line provided:

"""

# ========== 第 2 步：用 email_subject 组装 messages 列表 ==========

# 入参是主题行字符串；返回 Chat Completions / 兼容接口需要的 messages
def messages_for(email_subject):
    return [
        # system：角色与能力边界
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 具体主题行
        {"role": "user", "content": user_prompt_prefix + email_subject}
    ]

# 本次要生成邮件的主题行（改这里就能换场景）
email_subject = "Carrot Cake - New product launch in our Pastry Ltd. website"
# 调用上面的函数，得到发给本地接口的 messages
messages = messages_for(email_subject)

# ========== 第 3 步：通过 Ollama 的 OpenAI 兼容端点调用 llama3.2 ==========

# Ollama 对外暴露的 OpenAI 兼容基址（/v1）；需本机已启动 ollama serve
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建指向本地的客户端；api_key 对 Ollama 通常任意非空即可（此处原样保留 'ollama'）
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# 非流式请求：model 用本地 llama3.2，messages 与云端作业同一形状
response = ollama.chat.completions.create(model="llama3.2", messages=messages)
# 从返回结构里取出助手正文
email_message = response.choices[0].message.content

# ========== 第 4 步：以 Markdown 在笔记本中显示结果 ==========
display(Markdown(email_message))
